# IE Teachers Knowledge Graph Notebook
This Colab-first notebook walks through parsing IE teacher biographies, extracting entities, normalising labels, and building a NetworkX knowledge graph with friendly QA checkpoints.


In [ ]:
# Install dependencies (Restart & Run-All safe)
!pip install -q -r /content/ie-teachers-kg/requirements.txt


In [ ]:
# Global imports and deterministic setup
import os
import sys
import random
import json
from datetime import datetime
from collections import Counter

import numpy as np

random.seed(42)
np.random.seed(42)

REPO = "/content/ie-teachers-kg"
if REPO not in sys.path:
    sys.path.append(REPO)
SRC_PATH = os.path.join(REPO, "src")
if os.path.isdir(SRC_PATH) and SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

os.makedirs(os.path.join(REPO, "outputs"), exist_ok=True)
print("Repo path:", REPO)
print("Output dir:", os.path.join(REPO, "outputs"))


## Section A — Setup
We install the lightweight NLP stack (transformers + spaCy + NetworkX) and register the repo on `sys.path` so `src/` helpers are importable inside Colab.


## Section B — Load Data


In [ ]:
import pandas as pd

DATA_PATH = os.path.join(REPO, "data", "teachers_db_practice.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} professor bios from {DATA_PATH}")
df[['alias', 'area', 'position', 'full_info']].head()


## Section C — Pattern + NER fusion
We clean HTML, segment the bios into logical sections, run dual pattern parsers and multilingual NER on every bullet, and then fuse the evidence with a weighted scorer so we keep provenance, confidence, and traceability.

In [ ]:
from tqdm.auto import tqdm

from parsing import (
    clean_html,
    split_sections,
    sentences,
    iter_bullets,
    build_pattern_candidates,
    section_relation,
)
from ner_hf import load_pipelines, build_ner_candidates
from fusion import (
    Candidate,
    Scored,
    align_candidates,
    score_candidate,
    select_entities,
    FUSION_WEIGHTS,
)
from rules import (
    extract_courses,
    classify_org,
    year_bin,
    canon_org,
    canon_location,
    normalize_degree,
    ORG_ALIASES,
    LOCATION_ALIASES,
)
from normalize import normalize_name, cluster_and_canonicalize

pipes = load_pipelines()
print("Loaded pipelines:", list(pipes.keys()))

In [ ]:
from typing import Any, Dict, List


def lines_for_section(block: str) -> List[str]:
    '''Return bullet lines for a section, falling back to the raw block.'''

    if not block:
        return []
    lines = iter_bullets(block)
    if not lines:
        stripped = block.strip()
        return [stripped] if stripped else []
    return lines


def meta_payload(scored: Scored) -> Dict[str, Any]:
    '''Capture score, reasons, and sources for QA.'''

    merged = scored.merged
    return {
        'score': round(scored.score, 3),
        'reasons': scored.reasons,
        'sources': merged.meta.get('sources', []),
    }

In [ ]:
extraction_records = []
university_names = []
company_names = []
location_names = []
all_courses = []
org_type_counts = Counter()

THRESHOLD = 0.65

for idx, row in tqdm(enumerate(df.itertuples(index=False)), total=len(df)):
    html = getattr(row, 'full_info', '') or ''
    sections = split_sections(html)
    if not sections:
        sections = {'intro': clean_html(html)}
    prof_id = getattr(row, 'alias', f'prof_{idx}')
    record = {
        'prof_id': prof_id,
        'area': getattr(row, 'area', None),
        'position': getattr(row, 'position', None),
        'raw_sections': sections,
        'studies': [],
        'work': [],
        'courses': [],
        'locations': [],
    }

    for section_name, section_text in sections.items():
        block = (section_text or '').strip()
        if not block:
            continue
        for course in extract_courses(block):
            entry = {
                'course': course,
                'program': None,
                'source_section': section_name,
                'text_span': block,
            }
            record['courses'].append(entry)
            all_courses.append(course)
        lines = lines_for_section(block)
        pattern_cands = build_pattern_candidates(section_name, lines)
        ner_cands = []
        for line_idx, snippet in enumerate(lines):
            ner_cands.extend(build_ner_candidates(section_name, snippet, line_idx, pipes))
        aligned = align_candidates(pattern_cands, ner_cands)
        scored = [score_candidate(cand, FUSION_WEIGHTS) for cand in aligned]
        accepted = select_entities(scored, threshold=THRESHOLD)

        for scored_item in accepted:
            cand = scored_item.merged
            meta = meta_payload(scored_item)
            meta['section'] = section_name
            meta['text_span'] = cand.text_span
            relation = cand.relation
            if relation == 'worked_at':
                entry = {
                    'company': cand.org_raw,
                    'company_norm': cand.org_norm,
                    'company_canon': cand.org_canon,
                    'role': cand.role,
                    'location': cand.location_raw,
                    'location_norm': cand.location_norm,
                    'location_canon': cand.location_canon,
                    'start_year': cand.start_year,
                    'end_year': cand.end_year,
                    'end_year_text': cand.end_year_text,
                    'source_section': section_name,
                    'text_span': cand.text_span,
                    'meta': meta,
                }
                record['work'].append(entry)
                if cand.org_norm:
                    company_names.append(cand.org_norm)
                elif cand.org_canon:
                    company_names.append(cand.org_canon)
                if cand.location_norm:
                    location_names.append(cand.location_norm)
                elif cand.location_canon:
                    location_names.append(cand.location_canon)
                if cand.location_canon:
                    record['locations'].append({
                        'org': cand.org_canon or cand.org_norm or cand.org_raw,
                        'type': 'company',
                        'location': cand.location_canon,
                        'source_section': section_name,
                    })
            elif relation == 'studied_at':
                entry = {
                    'university': cand.org_raw,
                    'university_norm': cand.org_norm,
                    'university_canon': cand.org_canon,
                    'degree': cand.degree_text,
                    'degree_canon': cand.degree_level,
                    'field': cand.field,
                    'year': cand.year,
                    'year_bin': cand.year_bin,
                    'location': cand.location_raw,
                    'location_norm': cand.location_norm,
                    'location_canon': cand.location_canon,
                    'source_section': section_name,
                    'text_span': cand.text_span,
                    'meta': meta,
                }
                record['studies'].append(entry)
                if cand.org_norm:
                    university_names.append(cand.org_norm)
                elif cand.org_canon:
                    university_names.append(cand.org_canon)
                if cand.location_norm:
                    location_names.append(cand.location_norm)
                elif cand.location_canon:
                    location_names.append(cand.location_canon)
                if cand.location_canon:
                    record['locations'].append({
                        'org': cand.org_canon or cand.org_norm or cand.org_raw,
                        'type': 'university',
                        'location': cand.location_canon,
                        'source_section': section_name,
                    })
            if cand.org_type_guess != 'unknown':
                org_type_counts[cand.org_type_guess] += 1

    extraction_records.append(record)

print(f"Built {len(extraction_records)} extraction records")

## Section D — Normalisation & Canonical labels
We clean organisation and location names, cluster near-duplicates with RapidFuzz, and map degrees to a controlled vocabulary.


In [ ]:
CANON_DEGREES = {
    'PHD': 'PhD',
    'PHD.': 'PhD',
    'MBA': 'MBA',
    'MSC': 'MSc',
    'M.SC': 'MSc',
    'MS': 'MSc',
    'MA': 'MA',
    'M.A': 'MA',
    'BA': 'BA',
    'B.A': 'BA',
    'BSC': 'BSc',
    'B.SC': 'BSc',
    'MENG': 'MEng',
    'BENG': 'BEng',
}

uni_map = cluster_and_canonicalize(university_names, aliases=ORG_ALIASES)
comp_map = cluster_and_canonicalize(company_names, aliases=ORG_ALIASES)
loc_map = cluster_and_canonicalize(location_names, aliases=LOCATION_ALIASES)

print("University clusters (sample):", list(uni_map.items())[:8])
print("Company clusters (sample):", list(comp_map.items())[:8])
print("Location clusters (sample):", list(loc_map.items())[:8])
print("Org classification counts:", dict(org_type_counts))
print("Alias sanity checks:", {
    'Spaing': canon_location('Spaing'),
    'U. de Navarra': canon_org('U. de Navarra'),
    'MBA IE': canon_org('MBA IE'),
})


def canonical_degree(name):
    if not name:
        return None
    key = name.upper().replace('.', '')
    return CANON_DEGREES.get(key)


for record in extraction_records:
    for study in record['studies']:
        uni_label = study.get('university_canon') or study.get('university_norm') or study.get('university')
        if uni_label:
            uni_norm = canon_org(uni_label)
            study['university_norm'] = uni_norm
            study['university_canon'] = uni_map.get(uni_norm, uni_norm)
        loc_val = study.get('location_canon') or study.get('location_norm') or study.get('location')
        if loc_val:
            loc_norm = canon_location(loc_val)
            study['location_norm'] = loc_norm
            study['location_canon'] = loc_map.get(loc_norm, loc_norm)
        if not study.get('degree_canon'):
            study['degree_canon'] = canonical_degree(study.get('degree') or study.get('degree_text'))
        study['year_bin'] = study.get('year_bin') or year_bin(study.get('year'))
    for work in record['work']:
        comp_label = work.get('company_canon') or work.get('company_norm') or work.get('company')
        if comp_label:
            comp_norm = canon_org(comp_label)
            work['company_norm'] = comp_norm
            work['company_canon'] = comp_map.get(comp_norm, comp_norm)
        loc_val = work.get('location_canon') or work.get('location_norm') or work.get('location')
        if loc_val:
            loc_norm = canon_location(loc_val)
            work['location_norm'] = loc_norm
            work['location_canon'] = loc_map.get(loc_norm, loc_norm)
    for loc in record['locations']:
        loc_val = loc.get('location_canon') or loc.get('location')
        if loc_val:
            loc_norm = canon_location(loc_val)
            loc['location_norm'] = loc_norm
            loc['location_canon'] = loc_map.get(loc_norm, loc_norm)

print("Degree mapping:", CANON_DEGREES)

### Targeted QA checks
Two representative bios ensure the bullet parsers capture span details and canonical labels behave as expected.


In [ ]:

def find_record_by_companies(targets: set[str]):
    for rec in extraction_records:
        companies = {
            (work.get('company_canon') or work.get('company_norm') or work.get('company') or '').lower()
            for work in rec.get('work', [])
        }
        if targets.issubset({c for c in companies if c}):
            return rec
    return None

# Designer in Mexico/Spain
studio_targets = {name.lower() for name in ("A&M Studio", "Vidivixi", "Becquerel Capital", "The Hub")}
studio_record = find_record_by_companies(studio_targets)
assert studio_record, "Designer profile not found"
studio_work = []
expected_locations = {
    "a&m studio": "Spain",
    "vidivixi": "Mexico",
    "becquerel capital": "Mexico",
    "the hub": "Hong Kong",
}
for work in studio_record['work']:
    company = work.get('company_canon') or work.get('company_norm') or work.get('company')
    company_key = (company or '').lower()
    if company_key in studio_targets:
        loc = work.get('location_canon') or work.get('location')
        assert loc == expected_locations[company_key], f"Unexpected location for {company}: {loc}"
        studio_work.append({
            'company': company,
            'location': loc,
            'start_year': work.get('start_year'),
            'end_year': work.get('end_year') or work.get('end_year_text'),
            'score': work.get('meta', {}).get('score'),
            'reasons': work.get('meta', {}).get('reasons'),
        })
camberwell = next(
    (
        study
        for study in studio_record['studies']
        if study.get('university_canon') and 'camberwell' in study['university_canon'].lower()
    ),
    None,
)
assert camberwell, "Camberwell College degree missing"
print("Designer example:", json.dumps({'work': studio_work, 'degree': camberwell}, indent=2))

# BCG / Etisalat in UAE
mba_targets = {name.lower() for name in ("Boston Consulting Group", "Etisalat")}
mba_record = find_record_by_companies(mba_targets)
assert mba_record, "MBA/BCG profile not found"
mba_work = []
for work in mba_record['work']:
    company = work.get('company_canon') or work.get('company_norm') or work.get('company')
    company_key = (company or '').lower()
    if company_key in mba_targets:
        loc = work.get('location_canon') or work.get('location')
        assert loc == 'United Arab Emirates', f"Expected UAE for {company}, got {loc}"
        mba_work.append({
            'company': company,
            'location': loc,
            'score': work.get('meta', {}).get('score'),
            'reasons': work.get('meta', {}).get('reasons'),
        })

ie_degree = next(
    (
        study
        for study in mba_record['studies']
        if study.get('university_canon') == 'IE Business School'
    ),
    None,
)
upm_degree = next(
    (
        study
        for study in mba_record['studies']
        if study.get('university_canon') and 'politécnica de madrid' in study['university_canon'].lower()
    ),
    None,
)
assert ie_degree, "IE Business School degree missing"
assert upm_degree, "Universidad Politécnica de Madrid degree missing"
assert ie_degree.get('year') == 2015, f"Expected IE degree year 2015, got {ie_degree.get('year')}"
assert upm_degree.get('year') == 1999, f"Expected UPM year 1999, got {upm_degree.get('year')}"
print(
    "MBA example:",
    json.dumps({'work': mba_work, 'ie_degree': ie_degree, 'upm_degree': upm_degree}, indent=2),
)


### Fusion snippet QA
We stress-test the fusion stack with hand-crafted snippets that mirror the tricky bios (designer + BCG/Etisalat). Each snippet prints the accepted mentions with scores and reasons.

In [ ]:
from typing import Tuple


def inspect_lines(section_name: str, lines: list[str]) -> Tuple[list[dict], list[Scored]]:
    pattern = build_pattern_candidates(section_name, lines)
    ner = []
    for idx, line in enumerate(lines):
        ner.extend(build_ner_candidates(section_name, line, idx, pipes))
    aligned = align_candidates(pattern, ner)
    scored = [score_candidate(c, FUSION_WEIGHTS) for c in aligned]
    accepted = select_entities(scored, threshold=THRESHOLD)
    summary = []
    for item in accepted:
        cand = item.merged
        summary.append({
            'relation': cand.relation,
            'org': cand.org_canon or cand.org_norm or cand.org_raw,
            'location': cand.location_canon or cand.location_norm or cand.location_raw,
            'degree': cand.degree_level or cand.degree_text,
            'year': cand.year,
            'score': round(item.score, 3),
            'reasons': item.reasons,
        })
    return summary, accepted


designer_work_lines = [
    '• Creative Director, A&M Studio, Spaing, 2023–Present',
    '• Design Lead, Vidivixi, Mexico, 2017–2023',
    '• Senior Strategist, Becquerel Capital, Mexico, 2014–2017',
    '• Consultant, The Hub, Hong Kong, 2013–2014',
]
designer_study_lines = [
    '• Bachelor in Graphic Design, Camberwell College of Arts UAL, U.K., 2013',
]

work_summary, _ = inspect_lines('corporate_experience', designer_work_lines)
study_summary, _ = inspect_lines('academic_background', designer_study_lines)
expected_locations = {
    'A&M Studio': 'Spain',
    'Vidivixi': 'Mexico',
    'Becquerel Capital': 'Mexico',
    'The Hub': 'Hong Kong',
}
assert len(work_summary) == 4, work_summary
for entry in work_summary:
    assert entry['location'] == expected_locations[entry['org']], entry
assert any(item['degree'] in {'BSc', 'BA'} and item['location'] == 'United Kingdom' and item['year'] == 2013 for item in study_summary)
print('Designer fusion work:', json.dumps(work_summary, indent=2))
print('Designer fusion study:', json.dumps(study_summary, indent=2))

In [ ]:
bcg_lines = [
    '• Senior Partner, Boston Consulting Group (BCG), United Arab Emirates, 2018–Present',
    '• Vice President Strategy, Etisalat, United Arab Emirates, 2014–2018',
]
degree_lines = [
    '• International Executive MBA, MBA IE, Spain, 2015',
    '• Industrial Engineering, Universidad Politécnica de Madrid, Spain, 1999',
]

bcg_summary, _ = inspect_lines('corporate_experience', bcg_lines)
degree_summary, _ = inspect_lines('academic_background', degree_lines)
names = {entry['org'] for entry in bcg_summary}
assert any('Boston Consulting Group' in (name or '') for name in names), names
assert any('Etisalat' in (name or '') for name in names), names
for entry in bcg_summary:
    assert entry['location'] == 'United Arab Emirates'
assert any(entry['org'] == 'IE Business School' and entry['year'] == 2015 for entry in degree_summary)
assert any('Madrid' in (entry['org'] or '') and entry['year'] == 1999 for entry in degree_summary)
print('BCG fusion work:', json.dumps(bcg_summary, indent=2))
print('BCG fusion degrees:', json.dumps(degree_summary, indent=2))

## Section E — Graph building & artefacts


In [ ]:
from graph_utils import (
    new_graph,
    add_professor,
    add_university,
    add_company,
    add_course,
    add_degree,
    add_location,
    link_studied_at,
    link_worked_at,
    link_teaches,
    link_located_in,
    save_graph,
    top_k_by_degree,
)
import networkx as nx

G = new_graph()

for record in extraction_records:
    add_professor(G, record['prof_id'], area=record.get('area'), position=record.get('position'))
    for study in record.get('studies', []):
        univ = study.get('university_canon') or study.get('university_norm')
        if not univ:
            continue
        add_university(G, univ, location=study.get('location_canon'))
        if study.get('degree_canon'):
            add_degree(G, study['degree_canon'], field=study.get('field'))
        if study.get('location_canon'):
            link_located_in(G, univ, study['location_canon'], 'university')
        meta = study.get('meta', {})
        reasons = ' | '.join(meta.get('reasons', []))
        sources = ' | '.join(meta.get('sources', []))
        link_studied_at(
            G,
            record['prof_id'],
            univ,
            degree=study.get('degree_canon') or study.get('degree'),
            field=study.get('field'),
            year=study.get('year'),
            year_bin=study.get('year_bin'),
            source_section=study.get('source_section', 'unknown'),
            text_span=study.get('text_span'),
            confidence=meta.get('score'),
            reasons=reasons,
            sources=sources,
        )
    for work in record.get('work', []):
        comp = work.get('company_canon') or work.get('company_norm')
        if not comp:
            continue
        add_company(G, comp, location=work.get('location_canon'))
        if work.get('location_canon'):
            link_located_in(G, comp, work['location_canon'], 'company')
        meta = work.get('meta', {})
        reasons = ' | '.join(meta.get('reasons', []))
        sources = ' | '.join(meta.get('sources', []))
        link_worked_at(
            G,
            record['prof_id'],
            comp,
            role=work.get('role'),
            location=work.get('location_canon') or work.get('location'),
            start_year=work.get('start_year'),
            end_year=work.get('end_year') or work.get('end_year_text'),
            source_section=work.get('source_section', 'unknown'),
            text_span=work.get('text_span'),
            confidence=meta.get('score'),
            reasons=reasons,
            sources=sources,
        )
    for course in record.get('courses', []):
        name = course.get('course')
        if not name:
            continue
        add_course(G, name)
        link_teaches(
            G,
            record['prof_id'],
            name,
            program=course.get('program'),
            source_section=course.get('source_section', 'unknown'),
            text_span=course.get('text_span'),
        )

out_dir = os.path.join(REPO, 'outputs')
save_graph(G, out_dir)
print(f"Graph saved to {out_dir}")
print(nx.info(G))
print("Top universities:", top_k_by_degree(G, 'University'))
print("Top companies:", top_k_by_degree(G, 'Company'))

In [ ]:
import random
sampled = random.sample(extraction_records, min(10, len(extraction_records)))
for item in sampled:
    print(json.dumps(item, indent=2)[:1000])
    print('-' * 80)


## Section F — Quick visualisation


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

professors = [n for n, data in G.nodes(data=True) if data.get('type') == 'Professor']
selected = professors[:30]
sub_nodes = set(selected)
for prof in selected:
    sub_nodes.update(G.neighbors(prof))
H = G.subgraph(sub_nodes).copy()
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(H, seed=42)
color_map = []
for node in H:
    node_type = G.nodes[node].get('type')
    color_map.append({
        'Professor': '#1f77b4',
        'University': '#ff7f0e',
        'Company': '#2ca02c',
        'Course': '#d62728',
        'Location': '#9467bd',
        'Degree': '#8c564b',
    }.get(node_type, '#7f7f7f'))
nx.draw(H, pos, with_labels=False, node_color=color_map, node_size=120)
plt.title('Mini knowledge subgraph (first ~30 professors)')
plt.show()


## Section G — Export ZIP deliverable


In [ ]:
import shutil
import tempfile

stamp = datetime.utcnow().strftime('%Y%m%d')
zip_base = os.path.join(REPO, f'ie-teachers-kg_submit_{stamp}')
with tempfile.TemporaryDirectory() as tmp:
    targets = [
        ('notebooks', 'main.ipynb'),
        ('data', 'teachers_db_practice.csv'),
        ('outputs', 'nodes.csv'),
        ('outputs', 'edges.csv'),
        ('outputs', 'graph.gexf'),
        ('', 'requirements.txt'),
        ('', 'README.md'),
    ]
    for folder, fname in targets:
        src = os.path.join(REPO, folder, fname) if folder else os.path.join(REPO, fname)
        if os.path.exists(src):
            dst_dir = os.path.join(tmp, folder) if folder else tmp
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy2(src, os.path.join(dst_dir, fname))
    shutil.make_archive(zip_base, 'zip', tmp)

print(f"Created archive: {zip_base}.zip")


In [ ]:
try:
    from google.colab import files
    files.download(f"{zip_base}.zip")
except Exception as err:
    print("Download hint: run this cell in Colab to download the ZIP.")
    print(err)


## Section H — Documentation


**Pipeline pseudocode**
```
load CSV → iterate rows
  clean HTML → split sections
  run both NER models → merge spans
  attach regex degrees/courses + nearest locations
  normalise org/location strings via RapidFuzz clusters
  populate NetworkX graph with nodes + edges + provenance
persist nodes/edges/gexf → QA prints → build Colab ZIP deliverable
```


In [ ]:
degree_counter = Counter([
    study.get('degree_canon') or study.get('degree')
    for record in extraction_records for study in record['studies']
    if study.get('degree_canon') or study.get('degree')
])
course_counter = Counter(all_courses)
findings = [
    f"Processed {len(extraction_records)} professors with {len(G.nodes())} nodes and {len(G.edges())} edges in the KG.",
    f"Most common degrees: {degree_counter.most_common(3)}",
    f"Top courses mentioned: {course_counter.most_common(3)}",
    f"Top universities by degree centrality: {top_k_by_degree(G, 'University')[:3]}",
]
for item in findings:
    print(f"- {item}")
